In [1]:
# Prerequisites:
# pip install torch
# pip install docling_core
# pip install transformers

import json
import torch
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# Load images
image = load_image(r"other_resume_samples/4f1m9ybl7d1a1.png")

# Initialize processor and model
processor = AutoProcessor.from_pretrained("ibm-granite/granite-docling-258M")
model = AutoModelForVision2Seq.from_pretrained(
    "ibm-granite/granite-docling-258M",
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2" if DEVICE == "cuda" else "sdpa",
).to(DEVICE)

C:\Users\mosta\anaconda3\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "resume_sample_imgs_concat/CV - Karla Española.png"},
            {"type": "text", "text": """Parse the text from this resume and extract the following sections:
                - Work Experience (job titles, companies, dates, responsibilities)
                - Education (degrees, institutions, graduation dates)
                - Skills (technical skills, tools, certifications)

                Return the information in a structured JSON format."""}
        ]
    },
]

In [ ]:
# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=2048)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

print(f"DocTags: \n{doctags}\n")



# # Markdown:
# output_path_md = Path("out/") / "example.md"
# doc.save_as_markdown(output_path_md)

In [ ]:
# Populate document
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
# create a docling document
doc = DoclingDocument.load_from_doctags(doctags_doc, document_name="Document")

json_output = doc.export_to_json()

# Save to file
with open("docling_output.json", "w") as f:
    f.write(json_output)


# print(f"Markdown:\n{doc.export_to_json()}\n")
#
# # export as any format.
# Path("out/").mkdir(parents=True, exist_ok=True)
# # HTML:
# output_path_html = Path("out/") / "example.html"
# doc.save_as_html(output_path_html)

In [3]:
# print(trimmed_generated_ids)

tensor([[  6919,  21460, 100257]])


In [5]:
# # Try to parse as JSON
# try:
#     import re
#     cleaned = re.sub(r'```json\s*|\s*```', '', doctags)
#     parsed = json.loads(cleaned)
#     print(json.dumps(parsed, indent=2))
#
#     # Save to file
#     with open("parsed_resume.json", "w") as f:
#         json.dump(parsed, f, indent=2)
#     print("\n💾 Saved to: parsed_resume.json")
#
# except json.JSONDecodeError:
#     print(doctags)
#     print("\n⚠️ Output is not valid JSON")
#
#     # Save raw text
#     with open("parsed_resume.txt", "w") as f:
#         f.write(doctags)
#     print("💾 Saved raw text to: parsed_resume.txt")

Work Experience<|end_of_text|>

⚠️ Output is not valid JSON
💾 Saved raw text to: parsed_resume.txt
